# M11.8 · Bağlamsal haydutlar — Bayesçi doğrusal regresyon, LinUCB, Thompson

Durumlu haydutlar: her adımda bir bağlam (kullanıcı özellikleri) gelir, eylemlerin de öznitelikleri vardır (reklam konusu, anahtar kelimeler); geçiş yoktur, ardışık durumlar bağımsızdır. Ortalama ödül regresyonla yaklaşılır; Bayesçi doğrusal regresyonun kapalı form sonsalı ve öngörü dağılımı UCB'ye ve Thompson örneklemesine belirsizlik verir. Sherman–Morrison ile $O(d^2)$ güncelleme, ayrık ve ortak (x_s, x_a) öznitelikler, doğrusal olmayan ödüller için sinir ağı + Bayesçi son katman (NeuralLinear), kayıtlı veriden politika değerlendirme (IPS, SNIPS, DR, tekrar oynatma).

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m11-8-contextual-bandits.html](https://m-gonul.github.io/pytorch-lab/m11-8-contextual-bandits.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m11_8_contextual_bandits.py
CHECKS = {
 "b": "import random\nimport numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def blr():\n        rng = np.random.default_rng(0)\n        X = rng.normal(0, 1, (30, 4))\n        r = X @ np.array([1.0, -2, 0.5, 0]) + 0.3 * rng.normal(0, 1, 30)\n        m = need(ns, \"BayesLinReg\")(4, 2.0, 0.3)\n        for x, y in zip(X, r):\n            m.update(list(x), float(y))\n        Sig = np.linalg.inv(X.T @ X / 0.09 + np.eye(4) / 4)\n        mu = Sig @ X.T @ r / 0.09\n        close(np.array(m.Sigma), Sig, atol=1e-9, what=\"Σ (Sherman–Morrison = kapalı form)\")\n        q = [0.3, -1.0, 2.0, 0.5]\n        close(m.mean(q), float(np.array(q) @ mu), atol=1e-9, what=\"öngörü ortalaması\")\n        close(m.var(q), float(np.array(q) @ Sig @ np.array(q)), atol=1e-9, what=\"parametre belirsizliği x'Σx\")\n    R_.test(\"BayesLinReg\", blr)\n\n    def sim():\n        res = {p: sum(need(ns, \"news\")(5000, p, s) for s in range(4)) / 4 for p in (\"random\", \"ucb1\", \"linucb\")}\n        ensure(res[\"linucb\"] > res[\"ucb1\"] > res[\"random\"], f\"sıralama: {res}\")\n        R_.table(\"Haber önerisi, 5 makale, 5000 kullanıcı (4 tohum)\", [\"politika\", \"tıklama\", \"CTR\"],\n                 [[k, f\"{v:.0f}\", f\"{v / 5000:.3f}\"] for k, v in res.items()],\n                 note=\"UCB1 tek bir 'ortalama en iyi' makaleye yakınsar; LinUCB her kullanıcıya ilgisine göre makale seçer.\")\n    R_.test(\"LinUCB vs UCB1\", sim)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngü kuralları\", lambda: (max_loops(src, 0, func=\"posterior\"), max_loops(src, 1, func=\"sequential_posterior\"), max_loops(src, 0, func=\"predictive\"), max_loops(src, 0, func=\"joint_features\")))\n    rng = np.random.default_rng(3)\n    w = np.array([0.5, -1.0, 2.0])\n    X = rng.normal(0, 1, (200, 3))\n    r = X @ w + 0.5 * rng.normal(0, 1, 200)\n\n    def post():\n        mu, S = need(ns, \"posterior\")(np.array([[2.0]]), np.array([3.0]), 1.0, 1.0)\n        close(mu, np.array([1.2]), what=\"el hesabı μ\")\n        close(S, np.array([[0.2]]), what=\"el hesabı Σ\")\n        mu, S = ns[\"posterior\"](X, r, 1.0, 0.5)\n        mu2, S2 = need(ns, \"sequential_posterior\")(X, r, 1.0, 0.5)\n        close(mu2, mu, atol=1e-9, what=\"sıralı = toplu\")\n        close(S2, S, atol=1e-12, what=\"sıralı Σ\")\n        mean, var = need(ns, \"predictive\")(np.eye(3), mu, S, 0.5)\n        close(var, np.diag(S) + 0.25, what=\"öngörü varyansı\")\n        rows = []\n        for n in (5, 20, 200):\n            m_, S_ = ns[\"posterior\"](X[:n], r[:n], 1.0, 0.5)\n            rows.append([n, np.array2string(m_, precision=2), np.array2string(np.sqrt(np.diag(S_)), precision=3)])\n        R_.table(\"Sonsal daralması, gerçek w = (0.5, −1, 2)\", [\"n\", \"μ\", \"√diag Σ\"], rows, note=\"Belirsizlik ~ σ/√n hızında daralır; LinUCB'nin keşif bonusu bununla birlikte söner.\")\n    R_.test(\"posterior, sequential_posterior, predictive\", post)\n\n    def jf():\n        xs = np.array([1.0, 2.0])\n        XA = np.array([[1.0, 0, 3], [0, 1, 1]])\n        F = need(ns, \"joint_features\")(xs, XA)\n        close(F, np.array([[1, 0, 3, 2, 0, 6], [0, 1, 1, 0, 2, 2.0]]), what=\"dış çarpım öznitelikleri\")\n    R_.test(\"joint_features\", jf)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def blr():\n        g = torch.Generator().manual_seed(0)\n        Z = torch.randn(40, 3, generator=g, dtype=torch.float64)\n        A = torch.randint(0, 2, (40,), generator=g)\n        R = torch.randn(40, generator=g, dtype=torch.float64)\n        m1 = need(ns, \"BLR\")(2, 3, 1.0, 0.5)\n        for z, a, r in zip(Z, A, R):\n            m1.update(int(a), z, r)\n        m2 = ns[\"BLR\"](2, 3, 1.0, 0.5)\n        m2.refit(Z, A, R)\n        close(m1.P.numpy(), m2.P.numpy(), atol=1e-10, what=\"update = refit (kesinlik)\")\n        close(m1.b.numpy(), m2.b.numpy(), atol=1e-10, what=\"update = refit (b)\")\n        Zk = Z[A == 0]\n        mu = torch.linalg.solve(Zk.T @ Zk / 0.25 + torch.eye(3, dtype=torch.float64), Zk.T @ R[A == 0] / 0.25)\n        q = torch.tensor([1.0, -0.5, 2.0], dtype=torch.float64)\n        s = torch.stack([m2.sample(q, torch.Generator().manual_seed(i))[0] for i in range(4000)])\n        ensure(abs(float(s.mean() - q @ mu)) < 0.05, \"örneklerin ortalaması öngörü ortalaması\")\n    R_.test(\"BLR\", blr)\n\n    def cmp():\n        rl = [need(ns, \"run_lints\")(2000, s) for s in range(2)]\n        rn = [need(ns, \"run_neural_linear\")(2000, s) for s in range(2)]\n        late_l = sum(r[-1] - r[1499] for r in rl) / 2\n        late_n = sum(r[-1] - r[1499] for r in rn) / 2\n        ensure(late_n < 0.3 * late_l, f\"NeuralLinear XOR'u öğrenmeli (son 500 adım pişmanlığı {late_n:.1f} vs {late_l:.1f})\")\n        R_.table(\"XOR bağlamsal haydut, 2 tohum\", [\"yöntem\", \"pişmanlık t = 500\", \"t = 2000\", \"son 500 adım / adım\"],\n                 [[\"doğrusal Thompson (x, 1)\", f\"{sum(r[499] for r in rl) / 2:.0f}\", f\"{sum(r[-1] for r in rl) / 2:.0f}\", f\"{late_l / 500:.3f}\"],\n                  [\"NeuralLinear\", f\"{sum(r[499] for r in rn) / 2:.0f}\", f\"{sum(r[-1] for r in rn) / 2:.0f}\", f\"{late_n / 500:.3f}\"]],\n                 note=\"Doğrusal model XOR'u temsil edemez: belirsizliği sıfıra gider ama yanlış modele; pişmanlık doğrusal büyür. NeuralLinear öznitelikleri ağla öğrenir, belirsizliği yalnız son katmanda kapalı formda tutar (Riquelme ve ark. 2018).\")\n    R_.test(\"LinTS vs NeuralLinear\", cmp)\n",
 "e": "import torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef make(n, seed, env_seed=0, temp=0.8):\n    ge = torch.Generator().manual_seed(env_seed)\n    d, K = 5, 4\n    theta = torch.randn(K, d, generator=ge, dtype=torch.float64)\n    Wlog = temp * torch.randn(d, K, generator=ge, dtype=torch.float64)\n    g = torch.Generator().manual_seed(seed)\n    X = torch.randn(n, d, generator=g, dtype=torch.float64)\n    X[:, 0] = 1\n    mu = torch.sigmoid(X @ theta.T)\n    P = torch.softmax(X @ Wlog, 1)\n    A = torch.multinomial(P, 1, generator=g).squeeze(1)\n    R = (torch.rand(n, generator=g, dtype=torch.float64) < mu.gather(1, A[:, None]).squeeze(1)).double()\n    return X, A, R, P.gather(1, A[:, None]).squeeze(1), mu, theta\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: (max_loops(src, 0, func=\"estimates\"), max_loops(src, 0, func=\"fit_reward_model\"), max_loops(src, 0, func=\"policy_actions\")))\n\n    def frm():\n        X, A, R, p, mu, theta = make(3000, 1)\n        Th = need(ns, \"fit_reward_model\")(X, A, R, 4, 1.0)\n        for k in range(4):\n            Xk = X[A == k]\n            ref = torch.linalg.solve(Xk.T @ Xk + torch.eye(5, dtype=torch.float64), Xk.T @ R[A == k])\n            close(Th[k].numpy(), ref.numpy(), atol=1e-9, what=f\"eylem {k} sırt regresyonu\")\n    R_.test(\"fit_reward_model\", frm)\n\n    def ope():\n        theta = make(10, 0)[5]\n        g = torch.Generator().manual_seed(7)\n        W = torch.randn(200, 4, 5, generator=g, dtype=torch.float64)\n        W[:100] = theta[None] + 0.7 * W[:100]\n        errs = {k: [] for k in (\"DM\", \"IPS\", \"SNIPS\", \"DR\")}\n        found = {k: 0 for k in errs}\n        for s in range(40):\n            X, A, R, p, mu, _ = make(2000, 100 + s)\n            pa = need(ns, \"policy_actions\")(W, X)\n            ensure(tuple(pa.shape) == (200, 2000), \"şekil\")\n            true = mu.gather(1, pa.T).T.mean(1)\n            est = need(ns, \"estimates\")(W, X, A, R, p, ns[\"fit_reward_model\"](X, A, R, 4, 1.0))\n            for k, v in est.items():\n                errs[k].append(v - true)\n                found[k] += int(int(v.argmax()) == int(true.argmax()))\n        stats = {}\n        for k, v in errs.items():\n            E = torch.stack(v)\n            m, se = E.mean(0), E.std(0) / 40 ** 0.5\n            stats[k] = (float(E.pow(2).mean().sqrt()), float(m.abs().mean()), float((m.abs() / se).mean()))\n        ensure(stats[\"IPS\"][2] < 1.3 and stats[\"DR\"][2] < 1.3, f\"IPS ve DR yansız: |ortalama hata| / SE ≈ 0.8 olmalı ({stats['IPS'][2]:.2f}, {stats['DR'][2]:.2f})\")\n        ensure(stats[\"DM\"][2] > 2.0, \"yanlış belirtilmiş DM yanlı olmalı\")\n        ensure(stats[\"DR\"][0] < stats[\"IPS\"][0] and stats[\"SNIPS\"][0] < stats[\"IPS\"][0], \"DR ve SNIPS IPS'ten düşük RMSE\")\n        X, A, R, p, mu, _ = make(2000, 100)\n        Th = ns[\"fit_reward_model\"](X, A, R, 4, 1.0)\n        t_vec = bench(lambda: ns[\"estimates\"](W, X, A, R, p, Th), repeat=3, number=1)\n        t_loop = bench(lambda: [ns[\"estimates\"](W[i:i + 1], X, A, R, p, Th) for i in range(200)], repeat=1, number=1)\n        rows = [[k, f\"{v[0]:.4f}\", f\"{v[1]:.4f}\", f\"{v[2]:.2f}\", f\"{found[k]}/40\"] for k, v in stats.items()]\n        rows.append([\"süre: 200 politika tek tensör\", fmt_time(t_vec), \"\", \"\", \"\"])\n        rows.append([\"süre: politika döngüsü\", fmt_time(t_loop), \"\", \"\", \"\"])\n        R_.table(\"200 doğrusal politika × 40 kayıt kümesi (n = 2000, softmax kayıt politikası, sigmoid ödül)\", [\"kestirici\", \"RMSE\", \"|ort. hata|\", \"|ort. hata| / SE\", \"en iyi politikayı buldu\"], rows,\n                 note=\"Yansız bir kestiricide |ortalama hata| / SE ≈ √(2/π) ≈ 0.8 beklenir. DM'nin doğrusal ödül modeli sigmoid gerçeği yanlış belirtir: düşük varyanslı ama yanlı. IPS yansızdır, küçük p'lerde varyansı büyür. DR, DM'yi IPS ile düzeltir: yansız ve IPS'ten düşük varyanslı.\")\n    R_.test(\"DM, IPS, SNIPS, DR\", ope)\n",
 "x1": "import numpy as np\nfrom labcheck import close, need, ensure, fmt_time\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        A = rng.normal(0, 1, (4, 4))\n        A = A @ A.T + np.eye(4)\n        x = rng.normal(0, 1, 4)\n        close(need(ns, \"sm_update\")(np.linalg.inv(A), x), np.linalg.inv(A + np.outer(x, x)), atol=1e-10, what=\"Sherman–Morrison\")\n        rows = []\n        for d in (50, 200, 400):\n            ti, ts, err = need(ns, \"time_compare\")(d, 30)\n            ensure(err < 1e-8, \"aynı sonuç\")\n            rows.append([d, fmt_time(ti / 30), fmt_time(ts / 30), f\"{ti / ts:.1f}×\", f\"{err:.1e}\"])\n        R_.table(\"Güncelleme başına süre\", [\"d\", \"tam ters O(d³)\", \"Sherman–Morrison O(d²)\", \"hızlanma\", \"max fark\"], rows,\n                 note=\"Uzun koşularda yuvarlama hatası birikebilir: pratikte arada bir Cholesky ile yeniden hesaplanır.\")\n    R_.test(\"Sherman–Morrison\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rows, res = [], {}\n        for div in (True, False):\n            for pol in (\"greedy\", \"ts\"):\n                R = np.array([need(ns, \"linear_bandit\")(2000, pol, div, s) for s in range(40)])\n                res[(div, pol)] = R\n                rows.append([\"çeşitli N(0, I)\" if div else \"neredeyse sabit\", pol, f\"{R.mean():.1f}\", f\"{np.median(R):.1f}\", f\"{R.max():.0f}\"])\n        ensure(res[(True, \"greedy\")].mean() < 2 * res[(True, \"ts\")].mean(), \"çeşitli bağlamda açgözlü yeterli\")\n        ensure(res[(False, \"greedy\")].max() > 10 * res[(False, \"ts\")].max(), \"sabit bağlamda açgözlü bazı deneylerde takılmalı\")\n        R_.table(\"Doğrusal haydut, 2000 adım, 40 deney (pişmanlık)\", [\"bağlam\", \"politika\", \"ortalama\", \"ortanca\", \"en kötü\"], rows,\n                 note=\"Bağlamlar her yöne yeterince dağılıyorsa açgözlü seçim bile her parametreyi öğrenir (Bastani ve ark. 2021); bağlam sabite yakınsa problem bağlamsız haydutlaşır ve açgözlü takılır.\")\n    R_.test(\"bağlam çeşitliliği\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rows = []\n        for s in range(3):\n            X, A, R, th = need(ns, \"make_logs\")(25000, s)\n            ensure(X.shape == (25000, 5) and np.allclose(X.sum(1), 1), \"simpleks bağlamlar\")\n            est, cnt = need(ns, \"replay_linucb\")(X, A, R)\n            on = np.mean([need(ns, \"online_linucb\")(th, cnt, s * 10 + k) for k in range(5)])\n            rnd = float(np.mean(R))\n            ensure(abs(est - on) < 0.025, f\"tekrar oynatma ≈ çevrimiçi ({est:.3f} vs {on:.3f})\")\n            ensure(abs(cnt - 5000) < 300, \"kabul ≈ n/K\")\n            rows.append([s, cnt, f\"{rnd:.3f}\", f\"{est:.3f}\", f\"{on:.3f}\"])\n        R_.table(\"25 000 rastgele kayıt, 5 eylem\", [\"tohum\", \"kabul edilen\", \"kayıt politikası CTR\", \"LinUCB (tekrar oynatma)\", \"LinUCB (çevrimiçi, aynı adım sayısı)\"], rows,\n                 note=\"Kayıt politikası düzgün rastgele olduğu sürece tekrar oynatma yansızdır; bedeli veri: olayların ancak 1/K'sı kullanılır.\")\n    R_.test(\"tekrar oynatma\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m11_8_contextual_bandits as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Bağlamsal haydut:** durumlar $S$, her $s$ bir öznitelik vektörü $x_s$ (konum, yaş…); eylemler $A$, her $a$ bir $x_a$ (konu, anahtar kelimeler…). Geçiş yok — durumlar bağımsız gelir. Amaç: $\pi:x_s\to a$, $\mathbb E[r|x_s,x_a]$'yı büyüt. Uygulama: kişiselleştirilmiş reklam ve web içeriği (MSN haberleri bağlamsal haydutlarla tıklama oranında %26 artış bildirdi, 2017).

**Ödülü regresyonla öğren:** $\tilde R(s,a)=\tilde R(x)$, $x=(x_s,x_a)$; doğrusal $w^\top x$ ya da sinir ağı.

**Bayesçi doğrusal regresyon:** önsel $N(w|0,\lambda^2I)$, olabilirlik $N(r|w^\top x,\sigma^2)$ → sonsal Gauss:

$$\Sigma=\Big(\sigma^{-2}\sum_ix_ix_i^\top+\lambda^{-2}I\Big)^{-1},\qquad \mu=\sigma^{-2}\,\Sigma\sum_ix_ir_i.$$

Yeni $x$ için öngörü: $N\big(r\,\big|\,x^\top\mu,\;x^\top\Sigma x+\sigma^2\big)$ (ilk terim parametre belirsizliği, ikincisi gürültü).

- **UCB:** $x^\top\mu+c\sqrt{x^\top\Sigma x}$, $c=1+\sqrt{\ln(2/\delta)/2}$ (LinUCB).
- **Thompson:** $\tilde w\sim N(\mu,\Sigma)$, $a=\arg\max_a\tilde w^\top x_a$ (ya da $k$ örnek ortalaması).

Her adımda yalnız seçilen eylemin verisi gelir; $\Sigma^{-1}$'e $xx^\top/\sigma^2$ eklenir. **Sherman–Morrison:** $(A+uv^\top)^{-1}=A^{-1}-\dfrac{A^{-1}uv^\top A^{-1}}{1+v^\top A^{-1}u}$ — ters matrisi $O(d^2)$'de günceller.

**Kayıtlı veriden değerlendirme:** yeni bir politikayı canlıya almadan, eski politikanın kayıtlarıyla ($x$, $a$, $r$, $p(a|x)$) ölç: DM (ödül modeli), IPS ($\mathbb 1[\pi(x)=a]\,r/p$), SNIPS, DR; ya da tekrar oynatma (yalnız eylemler eşleşince say).

## 1 · Elle

$\Sigma=1/(x^2/\sigma^2+1/\lambda^2)=1/5$, $\mu=\Sigma\,xr/\sigma^2=0.2\cdot6=1.2$. Tek gözlem $w\approx r/x=1.5$ derdi; önsel 0'a doğru çekiyor (sırt regresyonu, $\lambda^{-2}\sigma^2=1$ ceza).

Sherman–Morrison: $I-\dfrac{xx^\top}{1+x^\top x}=I-\tfrac13\begin{pmatrix}1&1\\1&1\end{pmatrix}$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**1-B Bayesçi regresyon: önsel N(0, λ² = 1), σ² = 1, tek gözlem x = 2, r = 3**

- `sg`: Sonsal varyans Σ
- `mu`: Sonsal ortalama μ
- `pv`: x = 1'de yeni r'nin öngörü varyansı (gürültü dahil)

**LinUCB, δ = 0.05**

- `c`: $c=1+\sqrt{\ln(2/\delta)/2}$
- `ub`: x = 1'de üst sınır $x\mu+c\sqrt{x\Sigma x}$

**Sherman–Morrison: $A^{-1}=I_2$, $x=(1,1)$**

- `sm1`: $(A+xx^\top)^{-1}_{11}$
- `sm2`: $(A+xx^\top)^{-1}_{12}$

In [ ]:
cevaplar = dict(
    sg=None,
    mu=None,
    pv=None,
    c=None,
    ub=None,
    sm1=None,
    sm2=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"sg":[0.2,0],"mu":[1.2,0],"pv":[1.2,0],"c":[2.3581015157406195,0.000001],"ub":[2.254575057408263,0.000001],"sm1":[0.6666666666666666,1e-9],"sm2":[-0.3333333333333333,1e-9]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python. **Kişiselleştirilmiş haber:** K = 5 makale, her birinin gizli $\theta_a\in[0,0.5]^5$; kullanıcı bağlamı $x$ 5 boyutlu, negatif olmayan ve toplamı 1 (ilgi dağılımı); tıklama olasılığı $\theta_a^\top x$.

- `class BayesLinReg(d, lam=1.0, sigma=1.0)` — `Sigma` (liste listesi) ve `b` ($\sigma^{-2}\sum xr$) tutar. `update(x, r)`: Sherman–Morrison ile $\Sigma\leftarrow\Sigma-\dfrac{(\Sigma x)(\Sigma x)^\top}{\sigma^2+x^\top\Sigma x}$. `mean(x)` = $x^\top\Sigma b$, `var(x)` = $x^\top\Sigma x$.
- `news(T, policy, seed=0, c=1.0)` → toplam tıklama. `policy`: `"linucb"` (makale başına ayrı model, $\sigma=0.5$), `"ucb1"` (bağlamsız), `"random"`.

In [ ]:
%%writefile stage_b.py
import math
import random


class BayesLinReg:
    def __init__(self, d, lam=1.0, sigma=1.0):
        ...

    def update(self, x, r):
        ...

    def mean(self, x):
        ...

    def var(self, x):
        ...


def news(T, policy, seed=0, c=1.0):
    ...


print(news(2000, "linucb"), news(2000, "random"))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy.

- `posterior(X, r, lam, sigma)` → `(mu, Sigma)` (kapalı form, `np.linalg.solve`/`inv`).
- `sequential_posterior(X, r, lam, sigma)` — satır satır Sherman–Morrison (tek döngü); aynı sonuç.
- `predictive(Xq, mu, Sigma, sigma)` → `(mean, var)` her sorgu satırı için (gürültü dahil), döngüsüz (`einsum`).
- `joint_features(xs, XA)` — $x_s$ `(ds,)` ve eylem öznitelikleri `(K, da)` → `(K, ds·da)`: dış çarpım (etkileşim) öznitelikleri.

In [ ]:
%%writefile stage_c.py
import numpy as np


def posterior(X, r, lam, sigma):
    ...


def sequential_posterior(X, r, lam, sigma):
    ...


def predictive(Xq, mu, Sigma, sigma):
    ...


def joint_features(xs, XA):
    ...


X = np.array([[2.0]]); r = np.array([3.0])
print(posterior(X, r, 1.0, 1.0))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

**Doğrusal olmayan ödül:** bağlam $x\sim U[-1,1]^2$; eylem 0 → 0.6, eylem 1 → $\mathbb 1[x_1x_2>0]$, eylem 2 → $\mathbb 1[x_1x_2<0]$ (XOR), gürültü N(0, 0.1²). Verildi: `xor_means`.

- `class BLR(K, d, lam, sigma)` — eylem başına Bayesçi doğrusal regresyon, toplu tensörler: kesinlik `P (K, d, d)`, `b (K, d)`; `update(a, z, r)`, `refit(Z, A, R)` (sıfırdan, tüm veriyle), `sample(z, g)` → `(K,)` örneklenmiş ödüller (Cholesky).
- `run_lints(T, seed=0)` — öznitelik $z=(x,1)$; kümülatif pişmanlık listesi.
- `run_neural_linear(T, seed=0, retrain=50, iters=300)` — gövde 2→50→50 (ReLU) + doğrusal başlık (K çıkış), Adam 1e-2; her `retrain` adımda gövde+başlığı gözlenen (x, a, r) üzerinde MSE ile eğit (mini-batch 64), sonra BLR'yi $z=(\text{gövde}(x),1)$ özniteliklerinden yeniden kur. Eylem: BLR'den Thompson. `lam=1, sigma=0.5`.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn as nn


def xor_means(x):
    p = (x[:, 0] * x[:, 1] > 0).double()
    return torch.stack([torch.full_like(p, 0.6), p, 1 - p], 1)


class BLR:
    def __init__(self, K, d, lam=1.0, sigma=0.5):
        ...

    def update(self, a, z, r):
        ...

    def refit(self, Z, A, R):
        ...

    def sample(self, z, g):
        ...


def run_lints(T, seed=0):
    ...


def run_neural_linear(T, seed=0, retrain=50, iters=300):
    ...


print(run_lints(300)[-1])

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

**Politika dışı değerlendirme**, PyTorch; çok sayıda aday politika aynı anda.

Kayıtlar: `X (n, d)`, eylem `A (n,)`, ödül `R (n,)`, kayıt olasılığı `p (n,)`. Aday politikalar deterministik doğrusal: $\pi_W(x)=\arg\max_kW_k^\top x$, `W (P, K, d)`.

- `policy_actions(W, X)` → `(P, n)`.
- `fit_reward_model(X, A, R, K, lam=1.0)` — eylem başına sırt regresyonu (toplu `linalg.solve`) → `Theta (K, d)`; $\hat r(x,a)=\Theta_a^\top x$.
- `estimates(W, X, A, R, p, Theta)` → sözlük `{"DM", "IPS", "SNIPS", "DR"}`, her biri `(P,)`.

Politika döngüsü yok.

In [ ]:
%%writefile stage_e.py
import torch


def policy_actions(W, X):
    ...


def fit_reward_model(X, A, R, K, lam=1.0):
    ...


def estimates(W, X, A, R, p, Theta):
    ...


g = torch.Generator().manual_seed(0)
X = torch.randn(1000, 4, generator=g, dtype=torch.float64)
W = torch.randn(10, 3, 4, generator=g, dtype=torch.float64)
print(policy_actions(W, X).shape)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Sherman–Morrison: O(d³)'ten O(d²)'ye

- `sm_update(Ainv, x)` — NumPy, $(A+xx^\top)^{-1}$.
- `time_compare(d, n_updates=50, seed=0)` → `(süre_inv, süre_sm, max_fark)`: her güncellemede tam ters (`np.linalg.inv`) vs Sherman–Morrison.

In [ ]:
%%writefile extra_x1.py
import time
import numpy as np


def sm_update(Ainv, x):
    ...


def time_compare(d, n_updates=50, seed=0):
    ...


print(time_compare(100))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Açgözlü yeter mi? Bağlam çeşitliliği

- `linear_bandit(T, policy, diverse, seed=0)` — K = 2 eylem, d = 3, ayrık modeller (λ = 1, σ = 0.5), $\theta_a\sim N(0,I)$. Bağlam: `diverse=True` → $N(0,I)$; `False` → $(1,\;0.05\,\epsilon_2,\;0.05\,\epsilon_3)$ (neredeyse sabit). `policy ∈ {"greedy", "ts"}`. Toplam pişmanlık.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def linear_bandit(T, policy, diverse, seed=0):
    ...


print(linear_bandit(500, "greedy", True))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Tekrar oynatma ile çevrimdışı değerlendirme (Li ve ark. 2011)

**Tekrar oynatma:** kayıtlar düzgün rastgele politikayla toplandı. Değerlendirilen (öğrenen) politika olayları sırayla görür; seçtiği eylem kayıttakiyle aynıysa ödülü sayar **ve** bu olayla öğrenir, değilse olayı atlar. Kabul edilen olaylardaki ortalama ödül, politikanın çevrimiçi performansının yansız kestirimidir (Li ve ark. 2011).

- `make_logs(n, seed=0)` → `(X, A, R, theta)`: K = 5, d = 5, $x$ simpleks üzerinde (Dirichlet(1)), $p=\theta_a^\top x$, $\theta\sim U[0,0.5]$.
- `replay_linucb(X, A, R, alpha=1.0)` → `(ortalama ödül, kabul sayısı)`.
- `online_linucb(theta, T, seed=0, alpha=1.0)` → gerçek çevrimiçi ortalama ödül (aynı ortam, kendi bağlamları).

In [ ]:
%%writefile extra_x3.py
import math
import numpy as np


def make_logs(n, seed=0):
    ...


def replay_linucb(X, A, R, alpha=1.0):
    ...


def online_linucb(theta, T, seed=0, alpha=1.0):
    ...


X, A, R, th = make_logs(5000)
print(replay_linucb(X, A, R))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Li ve ark. (2010): LinUCB ile haber önerisi

[Li, Chu, Langford, Schapire — A Contextual-Bandit Approach to Personalized News Article Recommendation (WWW 2010)](https://arxiv.org/abs/1003.0146)

Algoritma 1 (ayrık LinUCB) ve Algoritma 2'yi (hibrit: paylaşılan + makaleye özel parametreler) uygula. Yahoo! R6 verisi (Webscope, başvuru gerektirir) ya da Open Bandit Dataset (Saito ve ark. 2020) üzerinde x3'teki tekrar oynatma yöntemiyle değerlendir; ε-greedy, UCB1 ve rastgele ile karşılaştır, α'yı tara.


**Adımlar**

1. b aşamasındaki Sherman–Morrison güncellemesini NumPy'a taşı.
2. Öğrenme ve dağıtım (deployment) aşamalarını ayır (makale Bölüm 5.2).

**Kabul ölçütleri**

- LinUCB'nin göreli CTR'si bağlamsız yöntemlerden %10–%15 daha yüksek (makale Tablo 2 nitel).
- Tekrar oynatmada kabul edilen olay sayısı ≈ toplam / K.

## Makale ödevi: Riquelme ve ark. (2018): Derin Bayesçi haydutlar karşılaştırması

[Riquelme, Tucker, Snoek — Deep Bayesian Bandits Showdown (ICLR 2018)](https://arxiv.org/abs/1802.09127)

Wheel bandit (δ ∈ {0.5, 0.7, 0.9, 0.95, 0.99}) ve en az iki gerçek veri kümesi (Mushroom, Statlog) üzerinde LinTS, NeuralLinear (d aşaması), dropout ve bootstrap ile Thompson'ı karşılaştır. Makalenin "sonsal belirsizliğin kalitesi ile kümülatif pişmanlık" bulgusunu tartış.


**Adımlar**

1. Ödülleri ölçekle; NeuralLinear'ı her t_f adımda yeniden eğit.
2. Her yöntem için 10 tohum, normalize pişmanlık.

**Kabul ölçütleri**

- δ büyüdükçe doğrusal yöntemler bozulur, NeuralLinear en sağlam yöntemlerdendir (makale Tablo 2–3 nitel).

## Kaynaklar

- [Li, Chu, Langford, Schapire — A Contextual-Bandit Approach to Personalized News Article Recommendation (WWW 2010)](https://arxiv.org/abs/1003.0146) — arXiv
- [Agrawal & Goyal — Thompson Sampling for Contextual Bandits with Linear Payoffs (ICML 2013)](https://arxiv.org/abs/1209.3352) — arXiv
- [Riquelme, Tucker, Snoek — Deep Bayesian Bandits Showdown (ICLR 2018)](https://arxiv.org/abs/1802.09127) — arXiv
- [Li, Chu, Langford, Wang — Unbiased Offline Evaluation of Contextual-bandit-based News Article Recommendation Algorithms (WSDM 2011)](https://arxiv.org/abs/1003.5956) — arXiv
- [Dudík, Langford, Li — Doubly Robust Policy Evaluation and Learning (ICML 2011)](https://arxiv.org/abs/1103.4601) — arXiv
- [Bastani, Bayati, Khosravi — Mostly Exploration-Free Algorithms for Contextual Bandits (Management Science 2021)](https://arxiv.org/abs/1704.09011) — arXiv
- [Microsoft Research Blog — Real-world interactive learning at cusp of enabling new class of applications (2017)](https://www.microsoft.com/en-us/research/blog/real-world-interactive-learning-cusp-enabling-new-class-applications/) — ©